# 03 控制与读取噪声建模（3 月）

目标：将 AWG 失真与测控链路响应通过等效参数纳入仿真。

说明：当前仓库尚无完整 AWG 失真链路模型，本 notebook 使用 control_scale、gate_duration、ou_sigma 做等效研究。


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'examples' / 'backend.yaml').exists():
            return p
    raise FileNotFoundError('???????????? pyproject.toml ? examples/backend.yaml?')

ROOT = find_project_root(Path.cwd())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from qsim.ui.notebook import run_workflow

BACKEND_PATH = ROOT / 'examples' / 'backend.yaml'
OUT_ROOT = ROOT / 'examples' / 'noise_simulation_tests' / 'runs' / 'roadmap_2026H1'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print('ROOT =', ROOT)
print('BACKEND_PATH =', BACKEND_PATH)
print('OUT_ROOT =', OUT_ROOT)


def run_case(tag: str, qasm_text: str, hardware: dict | None = None, noise: dict | None = None, engine: str = 'qutip'):
    out_dir = OUT_ROOT / tag
    return run_workflow(
        qasm_text=qasm_text,
        backend_path=str(BACKEND_PATH),
        out_dir=str(out_dir),
        hardware=hardware or {},
        noise=noise or {},
        engine=engine,
        persist_artifacts=True,
        export_dxf=False,
    )


def get_metric(result: dict, key: str, default: float = np.nan) -> float:
    obs = result.get('analysis', {}).get('observables', {}).get('values', {})
    return float(obs.get(key, default))

In [ ]:
QASM_1Q = """
OPENQASM 3;
qubit[1] q;
bit[1] c;
h q[0];
rz(0.5) q[0];
x q[0];
measure q[0] -> c[0];
"""

In [ ]:
scales = [0.6, 0.8, 1.0, 1.2, 1.4]
metric_scale = []
for s in scales:
    r = run_case(f'03_scale_{s:.1f}', QASM_1Q, hardware={'control_scale': s, 'gate_duration': 20.0}, noise={'model': 'markovian_lindblad', 't1': 100.0, 't2': 80.0})
    metric_scale.append(get_metric(r, 'final_p1'))

ou_sigmas = [0.0, 0.01, 0.02, 0.05]
metric_readout = []
for sig in ou_sigmas:
    r = run_case(f'03_ou_{sig:.3f}', QASM_1Q, hardware={'control_scale': 1.0, 'gate_duration': 20.0}, noise={'model': 'ou', 'ou_sigma': sig, 'ou_tau': 8.0})
    metric_readout.append(get_metric(r, 'final_p1'))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
ax[0].plot(scales, metric_scale, marker='o')
ax[0].set_title('AWG ?????????')
ax[0].set_xlabel('control_scale')
ax[0].set_ylabel('final_p1')
ax[0].grid(alpha=0.3)

ax[1].plot(ou_sigmas, metric_readout, marker='o', color='tab:red')
ax[1].set_title('??????????OU?')
ax[1].set_xlabel('ou_sigma')
ax[1].set_ylabel('final_p1')
ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()